<a href="https://colab.research.google.com/github/NicoleAki/Diabetes/blob/main/datapreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import kagglehub
path = kagglehub.dataset_download("mobeenfatimah/diabetes-risk-prediction-dataset-50k-patients")
df <- read_csv("data/diabetes_risk_prediction_dataset.csv", show_col_types = FALSE) |>
  clean_names()
dim(df)
glimpse(df)

ERROR: Error in parse(text = input): <text>:1:8: unexpected symbol
1: import kagglehub
           ^


In [ ]:
target <- "diabetes_risk"

if (!target %in% names(df)) {
  stop("Target not found. Columns are: ", paste(names(df), collapse = ", "))
}

In [ ]:
#data quality check
cat("Duplicate rows:", sum(duplicated(df)), "\n\n")

#missing values per column
missing_tbl <- df |>
  summarise(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(everything(), names_to = "column", values_to = "n_missing") |>
  mutate(pct_missing = round(100 * n_missing / nrow(df), 2)) |>
  arrange(desc(n_missing))

print(missing_tbl |> filter(n_missing > 0))
if (all(missing_tbl$n_missing == 0)) cat("No missing values.\n")

skim(df)

In [ ]:
#data types
cat("\nData types:\n")
print(df |> summarise(across(everything(), ~ class(.))) |>
  pivot_longer(everything(), names_to = "column", values_to = "data_type"))

In [ ]:
#data cleaning

In [ ]:
#target distribution
df |>
  count(.data[[target]]) |>
  mutate(pct = n / sum(n)) |>
  ggplot(aes(x = .data[[target]], y = n, fill = .data[[target]])) +
  geom_col(show.legend = FALSE) +
  geom_text(aes(label = percent(pct, accuracy = 0.1)), vjust = -0.4) +
  labs(title = "Target distribution", x = target, y = "Patients")

#### Target Distribution (`diabetes_risk`)

**Observations**
- The target is a **categorical variable** with severe **class imbalance**: High 73.2%, Moderate 25.9%, Low 0.9% (about 450 patients).
- A model that always predicts "High" would reach about 73% accuracy without learning anything, so accuracy alone is misleading.
- The "Low" class is too small for a model to learn reliably.

**Next steps**
- Set the class order to Low < Moderate < High.
- Use a **stratified** train/test split so every class appears in both sets.
- Evaluate with **macro-F1, per-class recall, and the confusion matrix** instead of accuracy.
- Use `class_weight = "balanced"`, or oversample on the **training set only**.
- Consider merging Low and Moderate into a binary target (High vs Not High) if "Low" remains unpredictable, and justify this in the report.

In [ ]:
#summary statistics for numeric columns
num_cols <- df |> select(where(is.numeric)) |> names()
cat_cols <- df |> select(where(is.factor)) |> names() |> setdiff(target)

df |>
  select(all_of(num_cols)) |>
  summarise(across(everything(), list(
    mean   = ~ mean(., na.rm = TRUE),
    sd     = ~ sd(., na.rm = TRUE),
    min    = ~ min(., na.rm = TRUE),
    median = ~ median(., na.rm = TRUE),
    max    = ~ max(., na.rm = TRUE)
  ))) |>
  pivot_longer(everything(),
               names_to = c("feature", "stat"),
               names_pattern = "(.*)_(mean|sd|min|median|max)$") |>
  pivot_wider(names_from = stat, values_from = value) |>
  mutate(across(where(is.numeric), ~ round(., 2)))

In [ ]:
options(repr.plot.width = 14, repr.plot.height = ceiling(length(num_cols) / 4) * 2.5)

df |>
  select(all_of(num_cols)) |>
  pivot_longer(everything(), names_to = "feature", values_to = "value") |>
  ggplot(aes(value)) +
  geom_histogram(bins = 30, fill = "#4c72b0", colour = "white") +
  facet_wrap(~ feature, scales = "free", ncol = 4) +
  labs(title = "Distribution of numeric features", x = NULL, y = NULL)


  #need to replot after removing outliers i.e nulls

#### Distribution of Numeric Features

**Observations**
- Most features (`blood_glucose`, `hb_a1c`, `hdl`, `ldl`, `triglycerides`, `total_cholesterol`, `height_cm`, `weight_kg`, `heart_rate`, `age`, etc.) are roughly **uniformly distributed**, which is typical of synthetic data where each column is sampled independently.
- `bmi` is **right-skewed**, with most values between 20 and 40 and a tail up to about 60.
- `diabetes_risk_score` is **left-skewed**, with most values between 60 and 100 and a spike at 100 (likely capped).
- The comb-like pattern in `age`, `heart_rate`, `blood_pressure_*` and `daily_water_intake_l` is a **binning artifact** from whole-number values, not a real pattern.
- `patient_id` is a unique identifier and carries no predictive information.

**Next steps**
- Drop `patient_id` before analysis and modelling.
- Apply a log transform to `bmi` only if a linear model (e.g. Logistic Regression) is used. Tree-based models don't need it.
- Check whether `diabetes_risk_score` is derived from the target. If it is, remove it to avoid **data leakage**.
- Use `binwidth = 1` for integer columns to remove the artifact.
- Check for impossible values (e.g. BMI below 12, very high glucose) before deciding whether to cap or remove outliers.

In [ ]:
#boxplots for numeric columns
df |>
  select(all_of(c(target, num_cols))) |>
  pivot_longer(-all_of(target), names_to = "feature", values_to = "value") |>
  ggplot(aes(x = .data[[target]], y = value, fill = .data[[target]])) +
  geom_boxplot(outlier.size = 0.5, show.legend = FALSE) +
  facet_wrap(~ feature, scales = "free_y", ncol = 4) +
  labs(title = "Numeric features by target", x = NULL, y = NULL)

#### Numeric Features by Target (Boxplots)

**Observations**
- **`diabetes_risk_score` separates the classes almost perfectly.** The ranges barely overlap (Low about 27-33, Moderate about 50-62, High about 73-89). It is almost certainly derived from the target, so it is a **data leakage** risk.
- **`hb_a1c` and `fasting_blood_sugar` are the strongest genuine predictors.** High has the highest values, Moderate sits in between, and Low is tightly clustered at the bottom. Low has many high outliers, so a few Low patients have high values, which suggests noise in how the data was generated.
- **Moderate effects** appear in `age`, `bmi`, `weight_kg`, `blood_pressure_systolic`, `blood_pressure_diastolic` and `total_cholesterol`. The High group has higher medians than Low, but the boxes overlap heavily.
- **Slight effects** appear in `height_cm`, `sleep_hours` and `waist_circumference_cm`: Low has slightly higher medians than High. Height and weight likely move together.
- **No visible difference** across classes for `blood_glucose`, `insulin_level`, `hdl`, `ldl`, `triglycerides`, `heart_rate`, `exercise_hours_per_week`, `daily_walking_minutes`, `daily_water_intake_l` and `patient_id`.
- `blood_glucose` showing no difference while `fasting_blood_sugar` and `hb_a1c` differ strongly suggests the target was built from a specific subset of columns rather than from clinically realistic relationships.

**Next steps**
- **Drop `patient_id`**, since it carries no signal.
- **Remove `diabetes_risk_score`** from the features, or train one model with it and one without, and report both. With it, accuracy will look near-perfect and mean little.
- **Keep** `hb_a1c`, `fasting_blood_sugar`, `age`, `bmi`, `weight_kg`, the blood pressure columns and `total_cholesterol` as the core features.
- **Test the flat features** with Kruskal-Wallis and effect size before dropping any, and use feature importance from the model to confirm.
- **Check combinations.** A weak feature alone can matter in an interaction (e.g. BMI with age), so tree-based models are worth including.
- **Handle the outliers** in `hb_a1c`, `fasting_blood_sugar` and `bmi` by keeping them, since they look genuine, and relying on tree models or scaling that tolerates them.
- **Reorder the x-axis** to Low, Moderate, High with `factor(..., levels = c("Low", "Moderate", "High"))` so the trend reads left to right.

In [ ]:
#correlation heatmap (pearson correlation))
options(repr.plot.width = 11, repr.plot.height = 9)

cor_mat <- cor(df |> select(all_of(num_cols)), use = "pairwise.complete.obs")

cor_mat |>
  as.data.frame() |>
  rownames_to_column("x") |>
  pivot_longer(-x, names_to = "y", values_to = "r") |>
  ggplot(aes(x, y, fill = r)) +
  geom_tile() +
  scale_fill_gradient2(low = "#c44e52", mid = "white", high = "#4c72b0", limits = c(-1, 1)) +
  labs(title = "Correlation matrix (numeric features)", x = NULL, y = NULL, fill = "r") +
  theme(axis.text.x = element_text(angle = 60, hjust = 1))

# Strongest pairs
cor_mat[upper.tri(cor_mat, diag = TRUE)] <- NA
as.data.frame(as.table(cor_mat)) |>
  drop_na() |>
  arrange(desc(abs(Freq))) |>
  head(10)

#### Correlation with the Target

**Observations**
- Pearson and Spearman correlations between numeric features and the target are weak.
- Features are mostly uniform and appear independently generated, so strong linear or monotonic relationships are not expected.

**Next steps**
- Use Kruskal-Wallis with an effect size (epsilon-squared) to compare features across the three risk levels, since the target is categorical.
- Use Cramér's V for categorical features.
- Fit a tree-based model and check feature importance to capture non-linear and interaction effects.

In [ ]:
#encode categorical features using one hot encoding
#perform spearman correlation for categorical features (for ordinal cat cols) and Cramer's V for nominal categorical features

